# Wang 5-Stack CNN evaluation — binary

Loads `best_model_wang_binary.keras` and the test set, generates predictions and computes the metrics used for comparison with the proposed binary CNN:

- **Global metrics**: accuracy, precision, recall and F1 for the rain class;
- **Classification report**: per-class precision, recall and F1;
- **2×2 confusion matrix**: absolute counts and row-normalized;
- **ROC curve** with AUC;
- **Precision-Recall curve** with Average Precision;
- **Per-class accuracy** bar chart;
- **Per-audio analysis**: majority vote over the 11 segments of each audio

Classes:
- `0`: no-rain
- `1`: rain

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_curve, auc, precision_recall_curve, average_precision_score,
    f1_score, accuracy_score, precision_score, recall_score,)
from data_pipeline import build_datasets, configure_gpu


In [ ]:
MODEL_PATH  = "best_model_wang_binary.keras"
CSV_PATH    = "Split70-15-15.csv"
BATCH_SIZE  = 32
CLASS_NAMES = ["no-rain", "rain"]
N_CLASSES   = 2


In [ ]:
# Load model and datasets
configure_gpu()

model = tf.keras.models.load_model(MODEL_PATH)
print(f"Modelo carregado: {MODEL_PATH}")
print(f"Total de parâmetros: {model.count_params():,}")

_, _, test_ds = build_datasets(CSV_PATH, batch_size=BATCH_SIZE, verbose=False)


In [ ]:
full_df = pd.read_csv(CSV_PATH)
test_df = full_df[full_df["split"] == "test"].reset_index(drop=True)

test_df["label_name"] = test_df["label"].map({0: "no-rain", 1: "rain"})

print(f"Total de amostras no test (segmentos): {len(test_df)}")
print(f"\nDistribuição por classe binária (segmentos):")
print(test_df["label_name"].value_counts().to_string())
print(f"\nDistribuição por classe original (segmentos):")
print(test_df["class_name"].value_counts().to_string())
print(f"\nDistribuição por classe (áudios únicos):")
print(test_df.groupby("label_name")["audio_id"].nunique().to_string())


In [ ]:
print("Rodando inferência no test set...")

y_probs_raw = model.predict(test_ds, verbose=1) 
y_prob      = y_probs_raw.squeeze()                
y_pred      = (y_prob >= 0.5).astype(int)
y_true      = test_df["label"].values.astype(int)

assert len(y_pred) == len(y_true), (
    f"Tamanhos diferentes! predict={len(y_pred)}, csv={len(y_true)}. "
    "Verifique se o test_ds está em shuffle=False.")

test_df["pred"]      = y_pred
test_df["prob_rain"] = y_prob        
test_df["prob_no_rain"] = 1 - y_prob 

print(f"\nDistribuição das predições:")
for i, name in enumerate(CLASS_NAMES):
    n = int((y_pred == i).sum())
    print(f"  - {name:>9}: {n}")


In [ ]:
# Global test metrics
test_accuracy   = accuracy_score(y_true, y_pred)
test_precision  = precision_score(y_true, y_pred, zero_division=0)
test_recall     = recall_score(y_true, y_pred, zero_division=0)
test_f1         = f1_score(y_true, y_pred, zero_division=0)

# Macro and weighted averages
test_prec_macro = precision_score(y_true, y_pred, average="macro",    zero_division=0)
test_rec_macro  = recall_score   (y_true, y_pred, average="macro",    zero_division=0)
test_f1_macro   = f1_score       (y_true, y_pred, average="macro",    zero_division=0)

test_prec_w = precision_score(y_true, y_pred, average="weighted", zero_division=0)
test_rec_w  = recall_score   (y_true, y_pred, average="weighted", zero_division=0)
test_f1_w   = f1_score       (y_true, y_pred, average="weighted", zero_division=0)

print("=" * 55)
print("MÉTRICAS FINAIS NO TEST SET — Wang 5-Stack CNN Binário")
print("=" * 55)
print(f"Accuracy:                     {test_accuracy:.4f}")
print()
print(f"Precision  (binary, rain):    {test_precision:.4f}")
print(f"Recall     (binary, rain):    {test_recall:.4f}")
print(f"F1-score   (binary, rain):    {test_f1:.4f}    <-- métrica principal")
print()
print(f"Precision  (macro):           {test_prec_macro:.4f}")
print(f"Recall     (macro):           {test_rec_macro:.4f}")
print(f"F1-score   (macro):           {test_f1_macro:.4f}")
print()
print(f"Precision  (weighted):        {test_prec_w:.4f}")
print(f"Recall     (weighted):        {test_rec_w:.4f}")
print(f"F1-score   (weighted):        {test_f1_w:.4f}")
print("=" * 55)

print("\nClassification report completo:")
print(classification_report(y_true, y_pred,
                            target_names=CLASS_NAMES, digits=4))


In [ ]:
# 2×2 confusion matrix (counts)
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=True,
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            annot_kws={"size": 16})
plt.xlabel("Predito", fontsize=12)
plt.ylabel("Real", fontsize=12)
plt.title("Matriz de Confusão (contagens) — Wang Binary — Test Set", fontsize=13)
plt.tight_layout()
plt.show()

print("Total por classe real:")
for i, name in enumerate(CLASS_NAMES):
    total   = int(cm[i].sum())
    correct = int(cm[i, i])
    print(f"  - {name:>9}: {correct}/{total} corretos ({correct/total*100:.2f}%)")


In [ ]:
# Row-normalized 2×2 confusion matrix (per-class recall)
cm_norm = confusion_matrix(y_true, y_pred, normalize="true")

plt.figure(figsize=(6, 5))
sns.heatmap(cm_norm, annot=True, fmt=".2%", cmap="Blues", cbar=True,
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            annot_kws={"size": 14})
plt.xlabel("Predito", fontsize=12)
plt.ylabel("Real", fontsize=12)
plt.title("Matriz de Confusão (normalizada) — Wang Binary — Test Set", fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# ROC curve and AUC
fpr, tpr, thresholds_roc = roc_curve(y_true, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, linewidth=2, color="#4878d0",
         label=f"Wang Binary  (AUC = {roc_auc:.4f})")
plt.plot([0, 1], [0, 1], "--", color="gray", label="Random (AUC = 0.5)")
plt.xlabel("False Positive Rate (1 - Especificidade)", fontsize=12)
plt.ylabel("True Positive Rate (Recall)", fontsize=12)
plt.title("Curva ROC — Wang 5-Stack CNN Binary — Test Set", fontsize=13)
plt.legend(loc="lower right", fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"AUC-ROC: {roc_auc:.4f}")


In [ ]:
# Precision-Recall curve and Average Precision (AP)
prec_curve, rec_curve, _ = precision_recall_curve(y_true, y_prob)
ap = average_precision_score(y_true, y_prob)

# Baseline: positive class ratio
baseline = y_true.mean()

plt.figure(figsize=(7, 6))
plt.plot(rec_curve, prec_curve, linewidth=2, color="#4878d0",
         label=f"Wang Binary  (AP = {ap:.4f})")
plt.axhline(baseline, linestyle="--", color="gray",
            label=f"Baseline (classe rain = {baseline:.2f})")
plt.xlabel("Recall", fontsize=12)
plt.ylabel("Precision", fontsize=12)
plt.title("Curva Precision-Recall — Wang 5-Stack CNN Binary — Test Set", fontsize=13)
plt.legend(loc="lower left", fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Average Precision (AP): {ap:.4f}")


In [ ]:
# Per-class accuracy bar chart
per_class_acc = []
for i in range(N_CLASSES):
    mask = (y_true == i)
    acc  = (y_pred[mask] == y_true[mask]).mean() if mask.sum() > 0 else 0.0
    per_class_acc.append(acc * 100)

plt.figure(figsize=(6, 5))
colors = ["#4878d0", "#ee854a"]
bars = plt.bar(CLASS_NAMES, per_class_acc, color=colors)
plt.ylim([0, 108])
plt.ylabel("Acerto (%)")
plt.title("Acurácia por classe — Wang Binary — Test Set (segmentos)")
for bar, pct in zip(bars, per_class_acc):
    plt.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 1, f"{pct:.2f}%",
             ha="center", fontsize=12)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# Per-audio analysis: majority vote of the 11 segments
audio_results = test_df.groupby("audio_id").agg(
    true_label    = ("label", "first"),
    pred_majority = ("pred",  lambda x: x.value_counts().idxmax()),
    label_name    = ("label_name", "first"),
    class_name    = ("class_name", "first"),
    n_segments    = ("pred",  "count"),
    agreement     = ("pred",  lambda x: x.value_counts().iloc[0] / len(x)),
    mean_prob_rain= ("prob_rain", "mean"),
).reset_index()

print(f"Total de áudios no test: {len(audio_results)}")
print(f"\nÁudios por classe binária:")
print(audio_results["label_name"].value_counts().to_string())

audio_acc      = accuracy_score(audio_results["true_label"],
                                audio_results["pred_majority"])
audio_f1       = f1_score(audio_results["true_label"],
                           audio_results["pred_majority"], zero_division=0)
audio_f1_macro = f1_score(audio_results["true_label"],
                           audio_results["pred_majority"],
                           average="macro", zero_division=0)

print("\n" + "=" * 55)
print("COMPARAÇÃO: segmento vs áudio — Wang Binary")
print("=" * 55)
print(f"Accuracy por SEGMENTO:                  {test_accuracy:.4f}")
print(f"Accuracy por ÁUDIO (voto majoritário):  {audio_acc:.4f}")
print(f"F1 (binary) por ÁUDIO:                  {audio_f1:.4f}")
print(f"F1 (macro)  por ÁUDIO:                  {audio_f1_macro:.4f}")
print("=" * 55)

# Per-audio confusion matrix
audio_cm = confusion_matrix(audio_results["true_label"],
                              audio_results["pred_majority"])

plt.figure(figsize=(6, 5))
sns.heatmap(audio_cm, annot=True, fmt="d", cmap="Greens", cbar=True,
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            annot_kws={"size": 16})
plt.xlabel("Predito (voto majoritário dos 11 segmentos)", fontsize=11)
plt.ylabel("Real", fontsize=12)
plt.title("Matriz de Confusão POR ÁUDIO — Wang Binary — Test Set", fontsize=13)
plt.tight_layout()
plt.show()

print(f"\nConcordância média entre os 11 segmentos: "
      f"{audio_results['agreement'].mean():.4f}")
print("(Próximo de 1.0 = predições consistentes; próximo de 0.5 = caos.)")

print("\nClassification report POR ÁUDIO:")
print(classification_report(audio_results["true_label"],
                            audio_results["pred_majority"],
                            target_names=CLASS_NAMES, digits=4))


In [ ]:
# Save misclassified segments for manual inspection
errors_df = test_df[test_df["label"] != test_df["pred"]].copy()
cols = ["path", "label", "pred", "class_name", "label_name", "audio_id",
        "prob_rain", "prob_no_rain"]
errors_df = errors_df[cols]
errors_df.to_csv("test_errors_wang_binary.csv", index=False)

print(f"Total de erros no test (segmentos): {len(errors_df)} / {len(test_df)}")
print(f"Taxa de erro: {len(errors_df)/len(test_df)*100:.2f}%")
print(f"Salvos em: test_errors_wang_binary.csv\n")

print("Distribuição dos erros por classe REAL (original):")
print(errors_df["class_name"].value_counts().to_string())


In [ ]:
np.savez("data_wang_binary.npz",
         y_true=y_true.astype(np.int8),
         y_probs=y_prob.astype(np.float32),
         audio_id=test_df["audio_id"].values.astype(str))
print("ok | segmentos:", len(y_true))